# ChagaSight — 2D Pathway Ablation  (`per_2d_fold.ipynb`)

### Purpose
Train **2D ViT only** (no 1D FM, no REPA) on one fold so you can compare:

| Model | Notebook | CSV output |
|-------|----------|------------|
| Hybrid (1D + 2D) | `10_train_fold_0.ipynb` | `fold0_results.csv` |
| **2D only** | **this notebook** | `fold0_2d_results.csv` |
| 1D only | `per_1d_fold.ipynb` | `fold0_1d_results.csv` |

### Design — what makes the comparison fair
- **Same fold** → identical train / val split → directly comparable metrics  
- **Same loss** → `AsymmetricBCE` (γ⁺=0, γ⁻=2, pos_weight=10) — identical to hybrid  
- **Same schedule** → P1: 2 000 iters (frozen), P2: 12 000 iters (full)  
- **Same eff.batch** → `BATCH=16 × accum=2 = 32` — same as hybrid  
- **Same pretrained weights** → `mae_2d_pretrained.pt` (same file as hybrid loads)  
- No REPA, no alignment loss (there is no FM pathway to align to)

In [1]:
import sys, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, WeightedRandomSampler
from tqdm import tqdm

# ── USER CONFIG ─────────────────────────────────────────────────────────────
FOLD              = 0        # use the SAME fold as per_1d_fold.ipynb
BATCH_SIZE        = 16
GRAD_ACCUM        = 2        # eff. batch = 32  — matches hybrid
PHASE1_ITERS      = 2_000    # head-only  (backbone frozen)
PHASE2_ITERS      = 24000   # full fine-tune
VAL_EVERY         = 8000      # matches per_1d_fold.ipynb
PHASE1_LR         = 2e-4
PHASE2_LR         = 2e-5
MAX_GRAD_NORM     = 1.0
NUM_WORKERS       = 2        # 0 on Windows if DLL errors
SEED              = 42

# ── PATHS ────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path(r"D:\IIT\L6\FYP\ChagaSight")
METADATA_CSV   = PROJECT_ROOT / "data/processed/metadata/combined_5fold.csv"
IMAGES_DIR     = PROJECT_ROOT / "data/processed/2d_images"
SIGNALS_DIR    = PROJECT_ROOT / "data/processed/1d_signals_100hz"   # still needed by ChagasDataset
MAE_CKPT       = PROJECT_ROOT / "checkpoints/mae_2d_pretrained.pt"
CKPT_DIR       = PROJECT_ROOT / "checkpoints"
BEST_PATH      = CKPT_DIR / f"fold{FOLD}_2d_best.pt"
OUT_CSV        = CKPT_DIR / f"fold{FOLD}_2d_results.csv"
OUT_PREDS      = CKPT_DIR / f"fold{FOLD}_2d_predictions.csv"

CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── REPRODUCIBILITY ──────────────────────────────────────────────────────────
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Device : {device}")
print(f"Fold   : {FOLD}  |  Eff.batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"P1 iters: {PHASE1_ITERS}  |  P2 iters: {PHASE2_ITERS}  |  Val every: {VAL_EVERY}")
for p, name in [(MAE_CKPT, "MAE ckpt"), (METADATA_CSV, "metadata")]:
    print(f"{name}: {'✓' if p.exists() else '✗  MISSING: ' + str(p)}")

Device : cuda
Fold   : 0  |  Eff.batch: 32
P1 iters: 2000  |  P2 iters: 24000  |  Val every: 8000
MAE ckpt: ✓
metadata: ✓


In [2]:
# ── Official PhysioNet scorer ────────────────────────────────────────────────
def compute_challenge_score(labels, outputs,
                             fraction_capacity=0.05,
                             num_permutations=10_000, seed=12345):
    labels  = np.asarray(labels,  dtype=np.float64)
    outputs = np.asarray(outputs, dtype=np.float64)
    num_instances = len(labels)
    capacity = int(fraction_capacity * num_instances)
    np.random.seed(seed)
    tp = np.zeros(num_permutations)
    for i in range(num_permutations):
        idx     = np.random.permutation(num_instances)
        ordered = labels[idx][np.argsort(outputs[idx])[::-1]]
        tp[i]   = ordered[:capacity].sum()
    tp_mean = tp.mean()
    return float(tp_mean / (tp_mean + (labels.sum() - tp_mean) + 1e-8))

print("✓ Scorer loaded")


✓ Scorer loaded


In [3]:
from src.training.dataset import ChagasDataset, custom_collate_fn

# Re-use the production ChagasDataset — it loads both image and signal.
# The 2D-only model will simply ignore batch['signal'].
train_ds = ChagasDataset(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    split='train', fold=FOLD,
    augment=True, use_soft_labels=True,
)
val_ds = ChagasDataset(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    split='val', fold=FOLD,
    augment=False, use_soft_labels=True,
)

# WeightedRandomSampler — 5× oversample positives (Van Santvliet et al.)
sample_weights = train_ds.get_sample_weights()
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, collate_fn=custom_collate_fn)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=custom_collate_fn)

print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")
print(f"Val  pos: {val_ds.df['label_hard'].sum():,}")


D:\IIT\L6\FYP\ChagaSight\src\training\dataset.py:89: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_csv)


✓ Loaded train fold 0: 292944 samples
  Datasets: {'ptbxl': 17439, 'samitrop': 1304, 'code15': 274201}
  Positive: 6552, Negative: 286392
✓ Loaded val fold 0: 73237 samples
  Datasets: {'ptbxl': 4360, 'samitrop': 327, 'code15': 68550}
  Positive: 1638, Negative: 71599
Train: 292,944  |  Val: 73,237
Val  pos: 1,638


D:\IIT\L6\FYP\ChagaSight\src\training\dataset.py:89: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_csv)


In [4]:
from src.models.vit_2d import ViT2D
from src.training.losses import AsymmetricBCELoss

class ViT2DClassifier(nn.Module):
    """2D backbone + single-pathway classification head."""
    def __init__(self):
        super().__init__()
        self.backbone = ViT2D(
            img_size=(24,2048), patch_size=(8,64), in_channels=3,
            embed_dim=768, depth=12, num_heads=12,
            mlp_ratio=4.0, dropout=0.1, use_aol=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)

    def forward(self, imgs):        # imgs: (B,3,24,2048) float [0,255] or [0,1]
        return self.head(self.backbone(imgs)).squeeze(-1)   # (B,)


model = ViT2DClassifier().to(device)

# Load MAE pretrained backbone
if MAE_CKPT.exists():
    model.backbone.load_mae_pretrained(str(MAE_CKPT))
    print("\n✓ MAE weights loaded")
else:
    print(f"\n⚠  {MAE_CKPT} not found — training from scratch")

n_all  = sum(p.numel() for p in model.parameters()) / 1e6
n_head = sum(p.numel() for p in model.head.parameters()) / 1e6
print(f"  Total: {n_all:.1f} M  |  Head: {n_head:.2f} M")

# AsymmetricBCE — SAME as hybrid  (γ⁺=0, γ⁻=2, pos_weight=10)
criterion = AsymmetricBCELoss(gamma_pos=0.0, gamma_neg=2.0, pos_weight=10.0)
scaler    = GradScaler()



 Loading MAE weights from: D:\IIT\L6\FYP\ChagaSight\checkpoints\mae_2d_pretrained.pt
   Top-level checkpoint keys: ['patch_embed', 'cls_token', 'pos_embed', 'encoder', 'encoder_norm']
   ✓ Flattened to 150 parameter tensors
     Trimmed CLS token: torch.Size([1, 97, 768]) → torch.Size([1, 96, 768])

 MAE weights loaded successfully:
   Loaded: 149 keys (was 5, now 145+ ✓)
   Missing: 0 keys (new components)
   Unexpected: 0 keys

   ✓ Transformer layers loaded: 144 weights
   Sample keys:
     - layers.0.self_attn.in_proj_weight: torch.Size([2304, 768])
     - layers.0.self_attn.in_proj_bias: torch.Size([2304])
     - layers.0.self_attn.out_proj.weight: torch.Size([768, 768])

   Critical components: 5/5 loaded
    All critical transformer weights loaded successfully!

✓ MAE weights loaded
  Total: 86.5 M  |  Head: 0.20 M


In [ ]:
# ── Iteration-based training loop ────────────────────────────────────────────

def make_infinite(loader):
    """Wrap a DataLoader into an infinite iterator."""
    while True:
        yield from loader


def validate(loader, n_perms=1000):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            imgs   = batch['image'].to(device, non_blocking=True)
            labels = batch['hard_label']
            with autocast(device_type='cuda'):
                probs = torch.sigmoid(model(imgs)).cpu().float().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.tolist())
    probs_arr  = np.array(all_probs,  dtype=np.float64)
    labels_arr = np.array(all_labels, dtype=np.int32)
    tpr5 = compute_challenge_score(labels_arr, probs_arr, num_permutations=n_perms)
    from sklearn.metrics import roc_auc_score, average_precision_score
    auroc = roc_auc_score(labels_arr, probs_arr)
    auprc = average_precision_score(labels_arr, probs_arr)
    model.train()
    return tpr5, auroc, auprc, probs_arr, labels_arr


def run_phase(phase_iters, lr, freeze_backbone, phase_name, resume_path=None):
    """One training phase.  Returns best (tpr5, val_history).

    Pass resume_path to enable mid-run checkpointing + resuming.
    A resume checkpoint is written every VAL_EVERY gradient steps; if it
    exists at startup, training continues from that point automatically.
    RNG state is saved/restored so there is no slow fast-forward on resume.
    Old checkpoints without RNG state are handled gracefully (no fast-forward).
    """
    if freeze_backbone:
        model.freeze_backbone()
        params = model.head.parameters()
    else:
        model.unfreeze_backbone()
        params = model.parameters()

    opt   = AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=phase_iters, eta_min=lr/20)

    best_tpr5   = 0.0
    global_step = 0          # gradient-update count (not accum_step)
    accum_loss  = 0.0
    val_history = []

    # ── Resume from checkpoint if available ──────────────────────────────────
    resume_path = Path(resume_path) if resume_path else None
    if resume_path and resume_path.exists():
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        opt.load_state_dict(ckpt['optimizer'])
        sched.load_state_dict(ckpt['scheduler'])
        scaler.load_state_dict(ckpt['scaler'])
        global_step = ckpt['global_step']
        best_tpr5   = ckpt['best_tpr5']
        val_history = ckpt.get('val_history', [])
        accum_loss  = ckpt.get('accum_loss', 0.0)
        # Restore RNG state if saved (new checkpoints); old checkpoints simply
        # start the data stream fresh from the resumed step — no fast-forward.
        if 'rng_state' in ckpt:
            torch.set_rng_state(ckpt['rng_state'])
            if 'cuda_rng_state' in ckpt and torch.cuda.is_available():
                torch.cuda.set_rng_state(ckpt['cuda_rng_state'])
            np.random.set_state(ckpt['np_rng_state'])
        print(f"  ↩  Resumed {phase_name} from iter {global_step}  "
              f"best_tpr5={best_tpr5:.4f}  ({resume_path.name})")
    # ─────────────────────────────────────────────────────────────────────────

    start_accum = global_step * GRAD_ACCUM
    train_iter  = make_infinite(train_loader)
    # No fast-forward loop — RNG state is restored from checkpoint instead.
    # (WeightedRandomSampler with replacement=True means exact batch replay
    #  is not possible anyway; the restored RNG keeps sampling fresh/correct.)

    model.train()
    opt.zero_grad(set_to_none=True)
    pbar = tqdm(total=phase_iters, initial=global_step,
                desc=phase_name, ncols=90)

    for accum_step in range(start_accum, phase_iters * GRAD_ACCUM):
        batch = next(train_iter)
        imgs   = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        with autocast(device_type='cuda'):
            logits = model(imgs)
            loss   = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()
        accum_loss += loss.item() * GRAD_ACCUM

        if (accum_step + 1) % GRAD_ACCUM == 0:
            global_step += 1
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad(set_to_none=True)
            pbar.set_postfix(loss=f"{accum_loss/global_step:.4f}", step=global_step)
            pbar.update(1)

            if global_step % VAL_EVERY == 0 or global_step == phase_iters:
                tpr5, auroc, auprc, _, _ = validate(val_loader, n_perms=1000)
                val_history.append((global_step, tpr5, auroc, auprc))
                flag = ""
                if tpr5 > best_tpr5:
                    best_tpr5 = tpr5
                    torch.save(model.state_dict(), BEST_PATH)
                    flag = "  ← BEST"
                tqdm.write(f"  [{phase_name}] iter {global_step:>6}  "
                           f"TPR@5%={tpr5:.4f}  AUROC={auroc:.4f}  AUPRC={auprc:.4f}{flag}")

                # ── Save resume checkpoint every VAL_EVERY steps ─────────────
                if resume_path:
                    torch.save({
                        'model':          model.state_dict(),
                        'optimizer':      opt.state_dict(),
                        'scheduler':      sched.state_dict(),
                        'scaler':         scaler.state_dict(),
                        'global_step':    global_step,
                        'best_tpr5':      best_tpr5,
                        'accum_loss':     accum_loss,
                        'val_history':    val_history,
                        'phase':          phase_name,
                        'rng_state':      torch.get_rng_state(),
                        'cuda_rng_state': torch.cuda.get_rng_state() if torch.cuda.is_available() else None,
                        'np_rng_state':   np.random.get_state(),
                    }, resume_path)
                # ─────────────────────────────────────────────────────────────

    pbar.close()
    return best_tpr5, val_history


print("✓ Training utilities ready")
print(f"Eff. batch = {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")

In [6]:
# ── PHASE 1 ── head-only  (backbone frozen) ─────────────────────────────────
print("\n" + "="*60)
print(f"PHASE 1 — {PHASE1_ITERS} iters  |  frozen backbone  |  LR={PHASE1_LR}")
print("="*60)

RESUME_P1 = CKPT_DIR / f"fold{FOLD}_2d_resume_p1.pt"

t0 = time.time()
best_p1, hist_p1 = run_phase(PHASE1_ITERS, PHASE1_LR, freeze_backbone=True,
                               phase_name="P1", resume_path=RESUME_P1)
print(f"\nPhase 1 done  ({(time.time()-t0)/60:.0f} min)  best TPR@5% = {best_p1:.4f}")


PHASE 1 — 2000 iters  |  frozen backbone  |  LR=0.0002
  ↩  Resumed P1 from iter 2000  best_tpr5=0.1624  (fold0_2d_resume_p1.pt)


P1: 100%|█████████████████████████████████████████████████████| 2000/2000 [00:00<?, ?it/s]



Phase 1 done  (11 min)  best TPR@5% = 0.1624


In [7]:
# ── PHASE 2 ── full fine-tune ────────────────────────────────────────────────
print("\n" + "="*60)
print(f"PHASE 2 — {PHASE2_ITERS} iters  |  full fine-tune  |  LR={PHASE2_LR}")
print("="*60)

RESUME_P2 = CKPT_DIR / f"fold{FOLD}_2d_resume_p2.pt"

t0 = time.time()
best_p2, hist_p2 = run_phase(PHASE2_ITERS, PHASE2_LR, freeze_backbone=False,
                               phase_name="P2", resume_path=RESUME_P2)
print(f"\nPhase 2 done  ({(time.time()-t0)/60:.0f} min)  best TPR@5% = {best_p2:.4f}")

best_overall = max(best_p1, best_p2)
print(f"\nOverall best TPR@5% = {best_overall:.4f}")


PHASE 2 — 24000 iters  |  full fine-tune  |  LR=2e-05
  ↩  Resumed P2 from iter 12000  best_tpr5=0.2490  (fold0_2d_resume_p2.pt)


KeyboardInterrupt: 

In [ ]:
# ── FINAL EVAL on val fold ───────────────────────────────────────────────────
model.load_state_dict(torch.load(BEST_PATH, map_location=device))

tpr5, auroc, auprc, probs_arr, labels_arr = validate(val_loader, n_perms=10_000)
n_pos   = int(labels_arr.sum())
n_total = len(labels_arr)

print("\n" + "="*60)
print(f"FOLD {FOLD}  2D-only  —  Final Results")
print("="*60)
print(f"  TPR@5%FPR  : {tpr5:.4f}   ← primary challenge metric")
print(f"  AUROC      : {auroc:.4f}")
print(f"  AUPRC      : {auprc:.4f}")
print(f"  N pos/total: {n_pos} / {n_total}")

# Save in exact same format as fold{N}_results.csv (hybrid)
result_df = pd.DataFrame([{
    "tpr_5pct": tpr5, "auroc": auroc, "auprc": auprc,
    "using_official": True, "num_permutations": 10000,
    "n_pos": n_pos, "n_total": n_total,
    "fold": FOLD, "quick_test": False,
}])
result_df.to_csv(OUT_CSV, index=False)
print(f"\n✓ Saved: {OUT_CSV}")


In [ ]:
# ── COMPARISON TABLE ─────────────────────────────────────────────────────────
print(f"\nAblation  —  Fold {FOLD}")
print(f"{'─'*55}")
print(f"{'Pathway':<22}  {'TPR@5%':>8}  {'AUROC':>8}  {'AUPRC':>8}")
print(f"{'─'*55}")
print(f"{'2D only  (this run)':<22}  {tpr5:>8.4f}  {auroc:>8.4f}  {auprc:>8.4f}")

hybrid_csv = CKPT_DIR / f"fold{FOLD}_results.csv"
if hybrid_csv.exists():
    h = pd.read_csv(hybrid_csv).iloc[0]
    print(f"{'Hybrid (1D+2D)':<22}  {h.tpr_5pct:>8.4f}  {h.auroc:>8.4f}  {h.auprc:>8.4f}")
    print(f"{'─'*55}")
    print(f"  Δ TPR@5%  (2D − hybrid): {tpr5 - h.tpr_5pct:+.4f}")

oneid_csv = CKPT_DIR / f"fold{FOLD}_1d_results.csv"
if oneid_csv.exists():
    d = pd.read_csv(oneid_csv).iloc[0]
    print(f"{'1D only  (per_1d_fold)':<22}  {d.tpr_5pct:>8.4f}  {d.auroc:>8.4f}  {d.auprc:>8.4f}")

print(f"{'─'*55}")
